# 🔮 QLoRA Fine-Tuning: Text-to-SQL with Mistral-7B

This notebook walks through the full pipeline:

1. **Data exploration** — inspect the `sql-create-context` dataset
2. **Prompt formatting** — compare template styles for different models
3. **QLoRA training** — 4-bit quantization + LoRA adapters
4. **Evaluation** — execution accuracy, BLEU, qualitative examples
5. **Inference** — interactive predictions

> **Hardware**: An A100 (40 GB) or equivalent is recommended.  
> A T4 (16 GB) works with reduced batch size.

In [ ]:
# ── Environment setup ─────────────────────────────────────────────
import os, sys, warnings
warnings.filterwarnings("ignore")

# Auto-detect repo root so imports work in Colab / local Jupyter
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from dotenv import load_dotenv
load_dotenv(os.path.join(REPO_ROOT, ".env"))

print(f"Repo root: {REPO_ROOT}")

---
## 1 · Data Exploration

In [ ]:
from datasets import load_dataset
import pandas as pd

ds = load_dataset("b-mc2/sql-create-context", split="train")
print(f"Total examples: {len(ds):,}")
print(f"Columns: {ds.column_names}")
ds[0]

In [ ]:
df = ds.to_pandas()
df["q_len"] = df["question"].str.len()
df["sql_len"] = df["answer"].str.len()
df["schema_len"] = df["context"].str.len()

df[["q_len", "sql_len", "schema_len"]].describe().round(0)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, col, color in zip(axes, ["q_len", "sql_len", "schema_len"],
                           ["steelblue", "coral", "seagreen"]):
    ax.hist(df[col], bins=60, color=color, edgecolor="white")
    ax.set_title(col.replace("_", " ").title())
    ax.set_xlabel("Characters")
plt.tight_layout()
plt.show()

---
## 2 · Prompt Formatting

We support several prompt styles depending on the base model.  
Below is a preview of each template.

In [ ]:
from src.data.prompt_templates import (
    format_prompt_generic,
    format_prompt_mistral,
    format_prompt_llama3,
    format_prompt_chatml,
)

sample = ds[42]
q, schema, sql = sample["question"], sample["context"], sample["answer"]

for name, fn in [
    ("Generic", format_prompt_generic),
    ("Mistral", format_prompt_mistral),
    ("Llama 3", format_prompt_llama3),
    ("ChatML", format_prompt_chatml),
]:
    print(f"\n{'═'*60}")
    print(f"  {name} template")
    print(f"{'═'*60}")
    print(fn(q, schema, sql))

---
## 3 · Prepare Dataset

In [ ]:
import yaml
from pathlib import Path

config_path = Path(REPO_ROOT) / "configs" / "training_config.yaml"
with open(config_path) as f:
    config = yaml.safe_load(f)

print("Model :", config["model"]["name"])
print("LoRA r:", config["lora"]["r"])
print("Epochs:", config["training"]["num_train_epochs"])

In [ ]:
from src.data.prepare_dataset import format_example, save_jsonl
from src.data.prompt_templates import get_formatter
from sklearn.model_selection import train_test_split

model_name = config["model"]["name"]
formatter = get_formatter(model_name)

formatted = [
    {"text": formatter(ex["question"], ex["context"], ex["answer"])}
    for ex in ds
]

train_data, test_data = train_test_split(
    formatted,
    test_size=config["dataset"].get("test_size", 0.1),
    random_state=42,
)

print(f"Train: {len(train_data):,} | Test: {len(test_data):,}")

---
## 4 · QLoRA Fine-Tuning

We load the base model in 4-bit, attach LoRA adapters, and train.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# ── Quantization ─────────────────────────────────────────────
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    token=os.getenv("HF_TOKEN"),
    attn_implementation="flash_attention_2",
)
model = prepare_model_for_kbit_training(model)

tokenizer = AutoTokenizer.from_pretrained(model_name, token=os.getenv("HF_TOKEN"))
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print(f"Model dtype: {model.dtype}")
print(f"GPU memory: {torch.cuda.memory_allocated()/1e9:.1f} GB")

In [ ]:
# ── LoRA config ───────────────────────────────────────────────
lora_cfg = config["lora"]

peft_config = LoraConfig(
    r=lora_cfg["r"],
    lora_alpha=lora_cfg["lora_alpha"],
    lora_dropout=lora_cfg["lora_dropout"],
    target_modules=lora_cfg["target_modules"],
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

In [ ]:
# ── Training ─────────────────────────────────────────────────
from datasets import Dataset
from trl import SFTConfig, SFTTrainer

train_dataset = Dataset.from_list(train_data)
eval_dataset  = Dataset.from_list(test_data[:500])  # small eval subset

sft_config = SFTConfig(
    output_dir="outputs/notebook-run",
    num_train_epochs=config["training"]["num_train_epochs"],
    per_device_train_batch_size=config["training"]["per_device_train_batch_size"],
    gradient_accumulation_steps=config["training"]["gradient_accumulation_steps"],
    learning_rate=float(config["training"]["learning_rate"]),
    lr_scheduler_type=config["training"]["lr_scheduler_type"],
    warmup_ratio=config["training"]["warmup_ratio"],
    max_seq_length=config["sft"]["max_seq_length"],
    dataset_text_field="text",
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=3,
    report_to="wandb",
    run_name="notebook-qlora-text2sql",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    args=sft_config,
)

print(f"Training {sft_config.num_train_epochs} epochs …")
trainer.train()

In [ ]:
# ── Save adapter ─────────────────────────────────────────────
adapter_dir = "outputs/notebook-run/final-adapter"
trainer.model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)
print(f"Adapter saved to {adapter_dir}")

---
## 5 · Evaluation

In [ ]:
from src.inference.predict import SQLPredictor

predictor = SQLPredictor(
    adapter_path=adapter_dir,
    base_model_name=model_name,
)

In [ ]:
# Quick spot-check on a few test examples
import random
random.seed(42)

samples = random.sample(list(ds), 5)
for s in samples:
    pred = predictor.predict(question=s["question"], schema=s["context"])
    print(f"Q: {s['question']}")
    print(f"Gold : {s['answer']}")
    print(f"Pred : {pred}")
    print(f"Match: {'✅' if pred.strip().lower() == s['answer'].strip().lower() else '❌'}")
    print("-" * 60)

---
## 6 · Interactive Demo

In [ ]:
schema = """CREATE TABLE employees (
    id INT PRIMARY KEY,
    name TEXT,
    department TEXT,
    salary REAL,
    hire_date TEXT
);"""

questions = [
    "How many employees are in the Engineering department?",
    "What is the average salary by department?",
    "List the top 5 highest paid employees.",
    "Who was hired most recently?",
]

for q in questions:
    sql = predictor.predict(question=q, schema=schema)
    print(f"Q: {q}")
    print(f"→ {sql}\n")

---
*End of notebook.*